# Evaluator Pattern — Build, Review, and Revise

This notebook demonstrates the evaluator MAS pattern: a `build_agent`
produces a result, a `review_agent` checks it against a checklist, and
the coordinator loops build to review to revise until the checklist
passes.

In [1]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPI
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code provided in this notebook, you'll need to have
Ollama installed on your local machine and have its LLM hosting
service running. To download Ollama, follow the instructions found on
this page: https://ollama.com/download. After downloading and
installing Ollama, you can start a service by opening a terminal and
running `ollama serve`.

In [2]:
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request


def ensure_ollama(host="http://localhost:11434", timeout=15):
    """Start Ollama if not already running and wait until responsive."""

    def _up():
        try:
            urllib.request.urlopen(f"{host}/api/tags", timeout=1)
            return True
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            return False

    if _up():
        return print(f"\u2713 Ollama already running at {host}")

    ollama_path = shutil.which("ollama")
    if ollama_path is None:
        for candidate in [
            "/teamspace/studios/this_studio/.local/bin/ollama",
            "/usr/local/bin/ollama",
            "/usr/bin/ollama",
        ]:
            if os.path.exists(candidate):
                ollama_path = candidate
                break
    if ollama_path is None:
        raise RuntimeError(
            "Could not find the ollama binary. Install with: "
            "curl -fsSL https://ollama.com/install.sh | sh",
        )

    print(f"Starting Ollama server ({ollama_path})...")
    subprocess.Popen(
        [ollama_path, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _up():
            return print(f"\u2713 Ollama up and running at {host}")
        time.sleep(0.5)

    raise RuntimeError(f"Ollama did not start within {timeout}s")


use_cloud = "OLLAMA_API_KEY" in os.environ
ensure_ollama() if not use_cloud else print("\u2713 Using Ollama Cloud")

✓ Using Ollama Cloud


In [3]:
model = "glm-5.3:cloud" if use_cloud else "qwen3:14b"
host = "https://ollama.com" if use_cloud else None

## Defining the Coding Standards

`check_code` is a house coding standard for a single function: it must
have a docstring, type hints on every parameter and the return value,
lines under 80 characters, and an exact required function name. This
mirrors what a real reviewer enforces: code that works isn't enough,
it also has to match the team's own conventions.

In [4]:
import ast

from llm_agents_from_scratch.tools.simple_function import SimpleFunctionTool

REQUIRED_NAME = "to_fahrenheit"
MAX_LINE_LEN = 79


def check_code(code: str) -> dict:
    """Check a function definition against the house coding standards."""
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return {"compliant": False, "issues": [f"syntax error: {e}"]}

    funcs = [n for n in ast.walk(tree) if isinstance(n, ast.FunctionDef)]
    if not funcs:
        return {"compliant": False, "issues": ["no function definition found"]}
    fn = funcs[0]

    issues = []
    if fn.name != REQUIRED_NAME:
        issues.append(
            f"function must be named exactly {REQUIRED_NAME!r}, "
            f"got {fn.name!r}",
        )
    if not ast.get_docstring(fn):
        issues.append("function is missing a docstring")
    if fn.returns is None:
        issues.append("missing a return type annotation")
    unannotated = [a.arg for a in fn.args.args if a.annotation is None]
    if unannotated:
        issues.append(
            f"missing type hints on parameter(s): {', '.join(unannotated)}",
        )
    long_lines = [
        i + 1
        for i, line in enumerate(code.splitlines())
        if len(line) > MAX_LINE_LEN
    ]
    if long_lines:
        issues.append(f"line(s) exceed {MAX_LINE_LEN} characters: {long_lines}")

    return {"compliant": not issues, "issues": issues}


check_code_tool = SimpleFunctionTool(func=check_code)

/home/nerdai/Projects/llm-agents-from-scratch/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Defining the Specialists

`build_agent` only ever sees the brief below, never the standards
above. `review_agent` never sees them either, in the sense that it
doesn't recite them; it just calls `check_code` and reports whatever
the tool actually returns.

In [5]:
from llm_agents_from_scratch import LLMAgent, LLMAgentBuilder
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.llms import OllamaLLM
from llm_agents_from_scratch.subagents import SubAgentSpec

llm = OllamaLLM(
    host=host,
    model=model,
    think=use_cloud,
    json_prompt_mode=use_cloud,
)

build_agent = SubAgentSpec(
    name="build_agent",
    description="Writes Python functions to a given brief.",
    builder=LLMAgentBuilder(llm=llm),
    max_steps=5,
)
review_agent = SubAgentSpec(
    name="review_agent",
    description="Checks Python functions against the house coding standards.",
    builder=LLMAgentBuilder(llm=llm, tools=[check_code_tool]),
    max_steps=5,
)

## Example — Converging on a Compliant Function

The brief below asks for a docstring, type hints, and a line-length
limit, but never says what the function must be named. `check_code`
still enforces an exact required name regardless, the kind of naming
convention a spec sheet states but an informal brief tends to leave
out.

In [6]:
brief = (
    "Write a Python function that converts a Celsius temperature to "
    "Fahrenheit. Include a docstring and type hints. Keep every line "
    "under 80 characters. Return only the function code."
)

coordinator = LLMAgent(llm=llm, subagents=[build_agent, review_agent])

task = Task(
    instruction=(
        f"Ask build_agent to: {brief} "
        "Then ask review_agent to check that exact code against the "
        "house coding standards: quote the full code in review_agent's "
        "task. If there are issues, ask build_agent to revise: quote "
        "the current code and the specific issues (quote them exactly) "
        "in build_agent's task. Then ask review_agent to check the "
        "revised code the same way. Repeat until compliant, up to 3 "
        "rounds. Call one subagent at a time and wait for its result "
        "before calling the next one. Report the final code and "
        "whether it's compliant."
    ),
)
handler = coordinator.run(task, max_steps=20)
result = await handler

In [7]:
print(result.content)

The revised code passed review — review_agent reports it is fully compliant with no issues (`{"compliant": true, "issues": []}`). The rename from `celsius_to_fahrenheit` to `to_fahrenheit` resolved the only issue, and everything else (docstring, type hints, line lengths, formula) was already in good shape. Since the code is compliant, no further revisions are needed and the revise/review cycle ends here after one revision round.

**Final code (verbatim):**

```python
def to_fahrenheit(celsius: float) -> float:
    """Convert a temperature from Celsius to Fahrenheit.

    Args:
        celsius: Temperature in degrees Celsius.

    Returns:
        The equivalent temperature in degrees Fahrenheit.
    """
    return celsius * 9 / 5 + 32
```

**Compliance status:** ✅ Compliant — review_agent confirmed the code passes the house coding standards with no issues found.

**Summary of the process:**
1. **Round 1 build:** build_agent produced the function named `celsius_to_fahrenheit`.
2. **Roun

`handler.rollout` shows why a revision was needed: `review_agent`'s
tool result names the specific naming mismatch, straight from
`check_code`'s own return value, not a guess, and the coordinator
relays that specific issue back to `build_agent` rather than just
asking it to try again.

In [8]:
print(handler.rollout)

=== Task Step Start ===

💬 assistant: My current instruction is 'Ask build_agent to: Write a Python function that converts a Celsius temperature to Fahrenheit. Include a docstring and type hints. Keep every line under 80 characters. Return only the function code. Then ask review_agent to check that exact code against the house coding standards: quote the full code in review_agent's task. If there are issues, ask build_agent to revise: quote the current code and the specific issues (quote them exactly) in build_agent's task. Then ask review_agent to check the revised code the same way. Repeat until compliant, up to 3 rounds. Call one subagent at a time and wait for its result before calling the next one. Report the final code and whether it's compliant.'

💬 assistant: I need to make the following tool call(s):

{
    "id_": "d5ad1642-e204-4a42-9ba1-a473bca4bd13",
    "tool_name": "from_scratch__use_subagent",
    "arguments": {
        "name": "build_agent",
        "task": "Write a Pyt

The 3-round cap is a safety net, not something this example needs.
`review_agent` never invents requirements, so once `build_agent`
finally sees the specific issue spelled out, satisfying it on the
next attempt is straightforward.